# 진자 행동 복제 (Behavior Cloning) — Colab

선생님 제어기(스윙업+세우기)의 시범을 시뮬레이터에서 모아, 신경망이 MSE 손실로 따라 하도록 학습합니다.
설명: 저장소의 `behavior_cloning/README.md`.

**사용법:** 위에서부터 순서대로 실행 (런타임 → 모두 실행). GPU는 필요 없습니다 (작은 신경망, 시뮬레이션은 CPU).

아래 두 `%%writefile` 셀은 저장소의 `simulator/rotary_model.py`, `behavior_cloning/bc_train.py`와 같은 내용입니다.

In [ ]:
!mkdir -p simulator behavior_cloning
!python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

## 1. 시뮬레이터 (가상 진자 + 선생님 제어기)

In [ ]:
%%writefile simulator/rotary_model.py
"""Rotary inverted pendulum (STEVAL-EDUKIT01) under acceleration control: plant, motor, controllers.

Shared by rotary_balance_realtime.py and rotary_swingup_realtime.py. The numbers and control
laws are the ones running on the real rig (swing-up/balance_control.py, swing-up/swingup_control.py,
swing-up/SwingupController/SwingupController.ino).

theta = pendulum angle from UPRIGHT (0 = balanced, +-pi = hanging), phi = arm angle,
u = phi'' [rad/s^2] (the host commands the arm acceleration).

    theta'' = (3g/2l) sin(theta) + b cos(theta) phi'' + sin(theta) cos(theta) phi'^2
              - 2 zeta wn theta' - Fc sign(theta')
    b = 3r/2l

Identified on our unit: wn = 7.499 rad/s (3g/2l = wn^2, l = 262 mm), zeta = 0.01172,
dry friction 2.705 deg/s of amplitude, b = 0.730 (r = 127 mm) from the open-loop test.

Motor = the firmware, not an ideal acceleration source:
    v += a dt           |a| <= 24000 pps^2, |v| <= 4000 pps   (target velocity)
    |v| < 30 pps        motor stopped (L6474 minimum run speed)
    |phi| >= 150 deg    latched stop
Measurement: 100 Hz loop, 4 ms from reading to the new command, 0.3 deg encoder,
arm angle in whole microsteps, theta' by filtered differencing.
"""
import math
from collections import deque

import numpy as np

# ---------------------------------------------------------------- plant (identified)
G = 9.81
WN, ZETA = 7.499, 0.01172
L = 3 * G / (2 * WN ** 2)                       # 0.262 m, uniform rod
A_G = WN ** 2                                   # 3g/2l [1/s^2]
C_D = 2 * ZETA * WN                             # viscous [1/s]
B_U = 0.730                                     # 3r/2l from the open-loop test
R_ARM = 2 * L * B_U / 3                         # 0.127 m
COULOMB_DPS = 2.705                             # amplitude lost per second to dry friction
F_C = math.radians(COULOMB_DPS) * WN ** 2 * (2 * math.pi / WN) / 4.0   # [rad/s^2]

# ---------------------------------------------------------------- motor (SwingupController.ino)
STEP_RAD = 2 * math.pi / 3200                   # 1/16 microstep
ACCEL_MAX = 24000 * STEP_RAD                    # 47.1 rad/s^2
SPEED_MAX = 4000 * STEP_RAD                     # 7.85 rad/s
SPEED_MIN = 30 * STEP_RAD                       # below this the motor does not run
ARM_LATCH = math.radians(150.0)

# ---------------------------------------------------------------- loop / sensor
CONTROL_HZ = 100.0
DELAY = 0.004                                   # s, reading -> new command
ENC_RAD = math.radians(0.3)                     # 1200 CPR
TAU_V = 0.01                                    # s, theta' filter
DT = 0.001                                      # plant step

# ---------------------------------------------------------------- controllers (hardware defaults)
POLES = [-6 + 6j, -6 - 6j, -1.5 + 1j, -1.5 - 1j]
SWING = dict(pump=30.0, ke=0.3, etarget=1.0, kpa=3.0, kda=5.0,
             catch=12.0, catch_w=4.0, catch_arm=60.0, fall=30.0, bias_tau=6.0)


def wrap(a):
    return (a + math.pi) % (2 * math.pi) - math.pi


def state_matrices(b=B_U):
    A = np.array([[0, 1, 0, 0], [A_G, -C_D, 0, 0], [0, 0, 0, 1], [0, 0, 0, 0]], float)
    B = np.array([[0], [b], [0], [1]], float)
    return A, B


def place4(poles=POLES, b=B_U):
    """Ackermann: K = e_n^T C^-1 phi(A), u = -K [theta, theta', phi, phi']."""
    A, B = state_matrices(b)
    ctrb = np.hstack([np.linalg.matrix_power(A, i) @ B for i in range(4)])
    phi = np.zeros_like(A)
    for c in np.real(np.poly(poles)):
        phi = phi @ A + c * np.eye(4)
    return np.linalg.solve(ctrb.T, np.eye(4)[:, -1]) @ phi


def place2(wn=12.0, zeta=0.7, b=B_U):
    """Rod only: s^2 + (c + b k2) s + (b k1 - a) = s^2 + 2 zeta wn s + wn^2. The arm is ignored."""
    return np.array([(wn ** 2 + A_G) / b, (2 * zeta * wn - C_D) / b, 0.0, 0.0])


def energy(th, w):
    """0 when upright at rest, -2 A_G (= -112) when hanging at rest."""
    return 0.5 * w * w + A_G * (math.cos(th) - 1)


class Plant:
    """x = [theta, theta', phi, v]; v is the firmware's target velocity, integrated from the command."""

    def __init__(self, theta0=0.0, b=B_U, coulomb=True, motor_limits=True):
        self.b, self.coulomb, self.motor_limits = b, coulomb, motor_limits
        self.reset(theta0)

    def reset(self, theta0=0.0):
        self.x = np.array([theta0, 0.0, 0.0, 0.0])
        self.a = 0.0                                # applied arm acceleration command
        self.t = 0.0
        self.latched = False

    def running(self, v):
        return (not self.motor_limits) or abs(v) >= SPEED_MIN

    def deriv(self, x, a):
        th, w, phi, v = x
        on = self.running(v)
        acc = a if on else 0.0
        phid = v if on else 0.0
        dw = (A_G * math.sin(th) + self.b * math.cos(th) * acc
              + math.sin(th) * math.cos(th) * phid * phid - C_D * w)
        if self.coulomb:
            dw -= F_C * math.tanh(w / 0.01)
        return np.array([w, dw, phid, a])

    def step(self):
        a = 0.0 if self.latched else self.a
        if self.motor_limits:
            a = max(-ACCEL_MAX, min(ACCEL_MAX, a))
        x = self.x
        k1 = self.deriv(x, a)
        k2 = self.deriv(x + DT / 2 * k1, a)
        k3 = self.deriv(x + DT / 2 * k2, a)
        k4 = self.deriv(x + DT * k3, a)
        self.x = x + DT / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        self.t += DT
        if self.motor_limits:
            self.x[3] = max(-SPEED_MAX, min(SPEED_MAX, self.x[3]))
            if abs(self.x[2]) >= ARM_LATCH and not self.latched:
                self.latched = True
                self.x[3] = 0.0

    @property
    def arm_rate(self):
        v = self.x[3]
        return v if self.running(v) else 0.0

    def kick(self, dw):
        self.x[1] += dw


class Sensor:
    """What the host sees: quantised encoder, whole microsteps, filtered theta'."""

    def __init__(self, realistic=True):
        self.realistic = realistic
        self.prev = None
        self.w = 0.0

    def reset(self):
        self.prev, self.w = None, 0.0

    def read(self, plant, offset=0.0):
        """offset = encoder zero error: the true vertical reads as `offset`."""
        th, w, phi, _ = plant.x
        if not self.realistic:
            return np.array([wrap(th + offset), w, phi, plant.arm_rate])
        m = round((th + offset) / ENC_RAD) * ENC_RAD
        ts = 1.0 / CONTROL_HZ
        if self.prev is not None:
            self.w += (ts / (TAU_V + ts)) * (wrap(m - self.prev) / ts - self.w)
        self.prev = m
        return np.array([wrap(m), self.w, round(phi / STEP_RAD) * STEP_RAD, plant.arm_rate])


class Balancer:
    """u = -K x. Gives up beyond `guard`."""

    def __init__(self, K, guard_deg=25.0):
        self.K, self.guard = np.asarray(K, float), math.radians(guard_deg)
        self.status = "OK"

    def reset(self):
        self.status = "OK"

    def __call__(self, z, t):
        if abs(z[0]) > self.guard:
            self.status = "FALLEN"
        return 0.0 if self.status != "OK" else float(-self.K @ z)


class SwingUp:
    """Same state machine as swing-up/swingup_control.py.

    swing   : +pump 0.15 s, -pump 0.15 s, then
              u = pump * clip(kE (E_target - E), 0, 1) * sign(theta' cos theta) - kda phi' - kpa phi
    catch   : |theta| < catch, |theta'| < catch_w, |phi| < catch_arm -> 4-state balancer,
              arm reference = arm angle at the catch, decaying with 3 s
    balance : slow upright-offset adaptation; beyond `fall` back to swing
    """

    def __init__(self, K, **kw):
        self.K = np.asarray(K, float)
        self.p = dict(SWING, **kw)
        self.gamma = abs(self.K[2]) / (self.K[0] * self.p["bias_tau"]) if self.p["bias_tau"] > 0 else 0.0
        self.reset()

    def reset(self, offset_guess=0.0):
        self.mode, self.phi_ref, self.b_hat = "swing", 0.0, offset_guess
        self.t_start = None
        self.t_catch, self.catches, self.falls = None, 0, 0
        self.E = float("nan")

    def __call__(self, z, t):
        p, ts = self.p, 1.0 / CONTROL_HZ
        if self.t_start is None:
            self.t_start = t
        tr = t - self.t_start
        th_raw, w, phi, phid = z
        th = wrap(th_raw - self.b_hat)
        self.E = energy(th, w)
        if (self.mode == "swing" and tr > 0.3 and abs(math.degrees(th)) < p["catch"]
                and abs(w) < p["catch_w"] and abs(math.degrees(phi)) < p["catch_arm"]):
            self.mode, self.phi_ref = "balance", phi
            self.catches += 1
            if self.t_catch is None:
                self.t_catch = tr
        elif self.mode == "balance" and abs(math.degrees(th)) > p["fall"]:
            self.mode = "swing"
            self.falls += 1
        if self.mode == "swing":
            if tr < 0.15:
                return p["pump"]
            if tr < 0.30:
                return -p["pump"]
            scale = min(1.0, max(0.0, p["ke"] * (p["etarget"] - self.E)))
            sgn = 1.0 if w * math.cos(th) > 0 else -1.0
            return p["pump"] * scale * sgn - p["kda"] * phid - p["kpa"] * phi
        self.phi_ref *= math.exp(-ts / 3.0)
        self.b_hat += self.gamma * (phi - self.phi_ref) * ts
        self.b_hat = max(-math.radians(5), min(math.radians(5), self.b_hat))
        return float(-self.K @ np.array([th, w, phi - self.phi_ref, phid]))


class Loop:
    """Plant at 1 kHz, controller at 100 Hz, each command applied DELAY after its reading."""

    def __init__(self, plant, controller, sensor=None, arm_limit_deg=140.0, offset=0.0):
        self.plant, self.ctrl = plant, controller
        self.sensor = sensor or Sensor()
        self.arm_limit = math.radians(arm_limit_deg)
        self.offset = offset                        # encoder zero error: true vertical reads as this
        self.on = True
        self.reset()

    def reset(self):
        self.next_ctrl = 0.0
        self.queue = deque()                        # (apply time, command)
        self.u = 0.0
        self.stop = ""
        self.sensor.reset()

    def _check_stop(self):
        pl = self.plant
        if pl.latched:
            self.stop = "FIRMWARE LATCH (150 deg)"
        elif abs(pl.x[2]) > self.arm_limit:
            self.stop = f"ARM LIMIT ({math.degrees(self.arm_limit):.0f} deg)"
        if self.stop:                               # host sends ZERO_VELOCITY + HARD_STOP
            self.queue.clear()
            pl.a, pl.x[3] = 0.0, 0.0

    def advance(self, n):
        pl = self.plant
        for _ in range(n):
            if pl.t >= self.next_ctrl - 1e-9:
                self.next_ctrl += 1.0 / CONTROL_HZ
                if not self.stop:
                    self._check_stop()
                z = self.sensor.read(pl, self.offset)
                u = self.ctrl(z, pl.t) if (self.on and not self.stop) else 0.0
                self.u = max(-ACCEL_MAX, min(ACCEL_MAX, u))
                if not self.stop:
                    lag = DELAY if self.sensor.realistic else 0.0
                    self.queue.append((pl.t + lag, self.u))
            while self.queue and pl.t >= self.queue[0][0] - 1e-9:
                pl.a = self.queue.popleft()[1]
            pl.step()


## 2. 학습 코드

In [ ]:
%%writefile behavior_cloning/bc_train.py
"""Behavior cloning of the swing-up + balance controller in the rotary pendulum simulator.

Teacher  = rotary_model.SwingUp (the controller that runs on the real rig).
Student  = small MLP: state -> arm acceleration command u [rad/s^2].
Loss     = MSE between the student's u and the teacher's u (both scaled by ACCEL_MAX).

    1. generate  teacher rollouts in simulator/rotary_model.py (randomised start, b, friction,
                 plus random pushes so the data contains falls and re-swings)
    2. train     MLP with MSE loss (default 1 epoch)
    3. evaluate  validation MSE, then the student alone drives the simulator (closed loop)

The state the student sees is what the teacher sees (Sensor.read):
    z = [theta (0 = upright), theta', phi (arm), phi']
theta enters as (sin, cos) so that +179 deg and -179 deg look alike to the network.

The teacher is not a pure function of z: it kicks open-loop for the first 0.3 s and switches
swing/balance with hysteresis. The kick samples are left out (no state can explain them);
the hysteresis band is what the student has to approximate.

    python behavior_cloning/bc_train.py                 # 1 epoch, prints losses + rollout
    python behavior_cloning/bc_train.py --epochs 30
"""
import argparse
import math
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "simulator"))
import rotary_model as rm  # noqa: E402

OUT = Path(__file__).resolve().parent / "output"


# ---------------------------------------------------------------- 1. teacher data
class Recorder:
    """Wraps the teacher: logs (z, u, mode) at every 100 Hz control tick."""

    def __init__(self, teacher):
        self.teacher, self.rows = teacher, []

    def __call__(self, z, t):
        u = self.teacher(z, t)
        tr = t - self.teacher.t_start
        u_sent = max(-rm.ACCEL_MAX, min(rm.ACCEL_MAX, u))   # Loop clips the same way
        self.rows.append((*z, u_sent, self.teacher.mode == "balance", tr >= 0.30))
        return u


def rollout_teacher(rng, duration):
    tilt = math.radians(rng.uniform(0.5, 3.0)) * rng.choice([-1, 1])
    plant = rm.Plant(math.pi + tilt, b=rm.B_U * rng.uniform(0.9, 1.1))
    rec = Recorder(rm.SwingUp(rm.place4()))
    lp = rm.Loop(plant, rec, rm.Sensor(True))
    f_c = rm.F_C
    rm.F_C = f_c * rng.uniform(0.5, 2.0)
    next_push = rng.uniform(4.0, 7.0)
    try:
        while plant.t < duration and not lp.stop:
            if plant.t >= next_push:                 # a hand slapping the rod
                plant.kick(rng.uniform(1.0, 4.0) * rng.choice([-1, 1]))
                next_push += rng.uniform(3.0, 6.0)
            lp.advance(10)                           # 10 x 1 ms = one control tick
    finally:
        rm.F_C = f_c
    d = np.array(rec.rows, dtype=np.float32)
    return d[d[:, 6] > 0]                            # drop the open-loop kick


def make_dataset(episodes, duration, seed):
    rng = np.random.default_rng(seed)
    eps = [rollout_teacher(rng, duration) for _ in range(episodes)]
    return eps


def features(z):
    """[theta, theta', phi, phi'] -> network input [sin, cos, theta', phi, phi'] (unscaled)."""
    z = np.asarray(z, dtype=np.float32)
    return np.stack([np.sin(z[..., 0]), np.cos(z[..., 0]), z[..., 1], z[..., 2], z[..., 3]], -1)


# ---------------------------------------------------------------- 2. student
class Policy(nn.Module):
    def __init__(self, mean, std, hidden=64):
        super().__init__()
        self.register_buffer("mean", torch.as_tensor(mean))
        self.register_buffer("std", torch.as_tensor(std))
        self.net = nn.Sequential(
            nn.Linear(5, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):                            # x: raw features -> u / ACCEL_MAX
        return self.net((x - self.mean) / self.std).squeeze(-1)

    @torch.no_grad()
    def act(self, z):
        x = torch.as_tensor(features(z)).unsqueeze(0)
        return float(self(x)) * rm.ACCEL_MAX


def run_epoch(model, X, Y, opt=None, batch=256):
    train = opt is not None
    model.train(train)
    idx = torch.randperm(len(X)) if train else torch.arange(len(X))
    total = 0.0
    with torch.set_grad_enabled(train):
        for i in range(0, len(X), batch):
            b = idx[i:i + batch]
            loss = nn.functional.mse_loss(model(X[b]), Y[b])
            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()
            total += loss.item() * len(b)
    return total / len(X)


# ---------------------------------------------------------------- 3. student drives the simulator
def rollout_student(model, duration, seed):
    rng = np.random.default_rng(seed)
    tilt = math.radians(rng.uniform(0.5, 3.0)) * rng.choice([-1, 1])
    plant = rm.Plant(math.pi + tilt)
    teacher = rm.SwingUp(rm.place4())                # used only for the 0.3 s open-loop kick
    start = {}

    def ctrl(z, t):
        start.setdefault("t", t)
        if t - start["t"] < 0.30:
            return teacher(z, t)
        return model.act(z)

    lp = rm.Loop(plant, ctrl, rm.Sensor(True))
    up_time, t_up = 0.0, None
    while plant.t < duration and not lp.stop:
        lp.advance(10)
        if abs(rm.wrap(plant.x[0])) < math.radians(12):
            up_time += 1.0 / rm.CONTROL_HZ
            t_up = plant.t if t_up is None else t_up
    return dict(upright_s=up_time, first_up_s=t_up, stop=lp.stop or "duration")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--episodes", type=int, default=40)
    ap.add_argument("--duration", type=float, default=15.0, help="s per teacher episode")
    ap.add_argument("--epochs", type=int, default=1)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no-rollout", action="store_true")
    args = ap.parse_args()
    torch.manual_seed(args.seed)
    OUT.mkdir(exist_ok=True)

    t0 = time.time()
    eps = make_dataset(args.episodes, args.duration, args.seed)
    n_val = max(1, len(eps) // 5)                    # split by episode, not by sample
    train, val = np.concatenate(eps[:-n_val]), np.concatenate(eps[-n_val:])
    print(f"teacher data: {len(train)} train / {len(val)} val samples from {len(eps)} episodes "
          f"({100 * train[:, 5].mean():.0f}% balancing) in {time.time() - t0:.1f} s")

    Xtr, Xva = (torch.as_tensor(features(d[:, :4])) for d in (train, val))
    Ytr, Yva = (torch.as_tensor(d[:, 4] / rm.ACCEL_MAX) for d in (train, val))
    model = Policy(Xtr.mean(0), Xtr.std(0) + 1e-6)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    print(f"before training: val MSE {run_epoch(model, Xva, Yva):.4f}  "
          f"(predict-zero baseline {float((Yva ** 2).mean()):.4f})")
    for ep in range(1, args.epochs + 1):
        tr = run_epoch(model, Xtr, Ytr, opt)
        va = run_epoch(model, Xva, Yva)
        print(f"epoch {ep:3d}: train MSE {tr:.4f}  val MSE {va:.4f}  "
              f"(val RMS error {math.sqrt(va) * rm.ACCEL_MAX:.1f} rad/s^2)")

    ckpt = OUT / "bc_policy.pt"
    torch.save(model.state_dict(), ckpt)
    print(f"saved {ckpt.relative_to(ROOT)}")

    if not args.no_rollout:
        r = rollout_student(model, 15.0, seed=1000 + args.seed)
        first = f"{r['first_up_s']:.2f} s" if r["first_up_s"] is not None else "never"
        print(f"student alone in the simulator (15 s): first reached upright {first}, "
              f"{r['upright_s']:.1f} s within 12 deg, ended by {r['stop']}")


if __name__ == "__main__":
    main()


## 3. 1 에포크 학습 (동작 확인, 약 20~40초)

In [ ]:
!python -u behavior_cloning/bc_train.py --epochs 1

## 4. (선택) 30 에포크 학습 — 학생 혼자 진자를 세우는지 확인

In [ ]:
!python -u behavior_cloning/bc_train.py --epochs 30

## 5. 학습된 신경망 내려받기

In [ ]:
from google.colab import files
files.download('behavior_cloning/output/bc_policy.pt')